# Week 2 Check-in
## STAT 390 | Spring 2026

---

## 1. Project Title

**Predicting ATP Grand Slam Match Outcomes Using Pre-Match History and Live First-Set Performance**

> *After the first set of a Grand Slam match is complete, what is the optimal way to combine a player's historical pre-match profile with their live first-set performance to predict who wins the match?*

Formally:
$$P(A\ \text{wins}) = f\bigl(w \cdot \text{first\_set\_features} + (1-w) \cdot \text{pre\_match\_features}\bigr)$$

The core question is finding the optimal weight $w \in [0, 1]$.

---
## 2. This Week's Goal

Build a **complete, reproducible data pipeline** from raw point-by-point tournament data to a single model-ready CSV:

1. Download all available ATP Grand Slam point-by-point data (Jeff Sackmann's repo)
2. Engineer first-set aggregate statistics per match from raw point sequences
3. Join first-set features with pre-match ATP metadata and clean the result
4. Complete exploratory data analysis to understand feature distributions and correlations before modeling

---
## 3. What I Completed

All four pipeline stages are done and producing clean output:

| Script | Output | Records |
|--------|--------|--------|
| `src/01_download_data.py` | `data/raw/slam_pbp/` (49 files) | 2011–2024 all four slams |
| `src/02_build_firstset_features.py` | `data/interim/firstset_features.csv` | **10,377 matches** |
| `src/03_join_and_clean.py` | `data/clean/tennis_model_ready.csv` | **3,638 matches × 52 features** |
| `src/04_explore_clean_data.py` | `data/plots/` (4 figures) | — |

**Non-trivial engineering challenges solved:**
- The raw data spans two incompatible formats (IBM Slamtracker 2011–2017 vs. Infosys MatchBeats 2018–present). Built format-detection logic with four fallback methods for extracting match winners and set scores.
- Initial join with ATP metadata returned only **30.8% match rate** because player order in slam_pbp is arbitrary. Fixed by switching to a sorted name-pair key → 100% join rate on filtered data.
- Applied random A/B assignment per row to balance the binary target (player order encodes no information).

---
## 4. Key Artifact: Dataset Summary + Correlation Analysis

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/clean/tennis_model_ready.csv')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Target balance: {df["A_won"].mean():.1%} class-1  (balanced by design)')
print(f'Date range: {df["year"].min()}–{df["year"].max()}')
print(f'Unique players: {len(set(df["player_A"].tolist() + df["player_B"].tolist())):,}')

In [ ]:
# Feature completeness summary
null_pct = df.isnull().mean().sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
print('Features with missing values:')
print(null_pct.to_string())

In [ ]:
# Top correlates with match outcome
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
correlations = df[numeric_cols].corr()['A_won'].drop('A_won').abs().sort_values(ascending=False)
print('Top 12 features by |correlation| with match outcome:')
print(correlations.head(12).round(3).to_string())

In [ ]:
# Display the EDA figures
from IPython.display import Image, display
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

img1 = plt.imread('../data/plots/02_correlation_heatmap.png')
img2 = plt.imread('../data/plots/03_firstset_stats_by_outcome.png')

axes[0].imshow(img1)
axes[0].axis('off')
axes[0].set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')

axes[1].imshow(img2)
axes[1].axis('off')
axes[1].set_title('First-Set Stats by Match Outcome', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Set-1 winner → match winner rate by surface
s1_win_col = 's1_A_win' if 's1_A_win' in df.columns else 's1_winner_A'

# If the column exists, compute win-from-set1 rates
if s1_win_col in df.columns:
    overall = (df[s1_win_col] == df['A_won']).mean()
    by_surface = df.groupby('surface').apply(
        lambda g: (g[s1_win_col] == g['A_won']).mean()
    ).sort_values(ascending=False)
    print(f'Set-1 winner wins the match: {overall:.1%} overall')
    print('By surface:')
    print(by_surface.round(3).to_string())
else:
    print('Set-1 win column not found — check column names:', [c for c in df.columns if 's1' in c][:10])

### Key Findings

| Finding | Value | Implication |
|---------|-------|-------------|
| Set-1 winner wins match | **77.8%** overall | Strong in-match signal — first set matters a lot |
| Higher-ranked player wins | **72.9%** | Pre-match ranking is a meaningful baseline |
| `s1_return_pts_won_pct` | r = **0.436** | Best continuous first-set predictor |
| `s1_first_srv_pct` | r = **0.063** | Getting the serve in ≠ winning the match |
| `avg_rally_length` | r ≈ **0.000** | Dropped — 44% null with no signal |

Serving well matters less than *winning* the serve point and *winning points on the opponent's serve* (return game). This directly shapes which first-set features will go into Stage 2 modeling.

---
## 5. Biggest Blocker

**Low initial join rate (30.8%) when merging slam_pbp with ATP metadata.**

The slam_pbp files assign `player1`/`player2` order arbitrarily (whoever is listed first in the source data), while the ATP metadata has a single canonical row per match with a specific player ordering. A directional join on `(player_A, player_B)` silently failed ~50% of the time when the order was reversed.

**Resolution:** Switched to a *sorted name-pair key* — for every match, sort both player names alphabetically and join on the resulting `(name_low, name_high)` pair. This is order-independent and brought the match rate to essentially 100% after filtering to ATP men's singles.

---
## 6. Plan for Next Week

**Goal: Build and evaluate Stages 1 and 2 baseline models.**

1. **Stage 1 — Pre-match logistic regression:**
   - Features: `rank_diff`, `rank_pts_A/B`, `age_diff`, `ht_diff`, `surface` (encoded), `hand_A/B`
   - This is the "historical only" benchmark — equivalent to a pre-match betting line
   - Expected accuracy: ~66–68%

2. **Stage 2 — First-set logistic regression:**
   - Features: `s1_margin`, `s1_A_return_pts_won_pct`, `s1_A_first_srv_won_pct`, `s1_A_aces`, `s1_A_df`, `s1_A_win`, `s1_A_ue`, `s1_A_bp_faced`, `s1_A_bp_saved_pct`
   - Missing serve stats (16–18% null): impute with per-surface-per-year median
   - Expected accuracy: ~78–80% (first set already contains enormous information)

3. Evaluation framework: 5-fold CV on 2011–2021 training set + held-out 2022–2023 test set, reporting accuracy, Brier score, and ROC-AUC

Deliverable: `notebooks/week3_baseline_models.ipynb`

---
## 7. Help Needed

**Imputation strategy for null serve statistics:**

About 16–18% of rows are missing `s1_first_srv_pct`, `s1_first_srv_won_pct`, and `s1_second_srv_won_pct`. These nulls are concentrated in **early years (2011–2013)** where IBM Slamtracker did not record per-point serve-type indicators — so the missing data is **not random** (it's MCAR within a year, but MAR across time).

My current plan: impute with the per-surface-per-year median. But given that we're predicting *match outcomes* and the null rows are from older, potentially different-era tennis, I'd welcome advice on whether:
- Simply **dropping the null rows** for Stage 2/3 would be cleaner than imputing
- Using a **missingness indicator** alongside median imputation is worth the extra complexity
- There's a smarter era-aware approach I should consider